# 01 データ取得・年別結合

このノートブック1本で、必要に応じたAPI取得から年別CSVの保存まで実行します。
対象年は `settings.py`、取得の有無と入力フォルダは次のセルで設定します。

- 取得済みブロックがある：`DOWNLOAD_MISSING = False`（APIキー不要）。
- APIから取得する：`DOWNLOAD_MISSING = True` にしてAPIキーを登録します。

設定後は上から順に実行してください。全期間の一括メモリ読み込みは行いません。
出力は `settings.py` の `YEAR_DIR` にある `trade_{年}.csv.gz` です。


## 1. 設定と準備


In [ ]:
import time
from pathlib import Path

import pandas as pd
from IPython.display import display

from settings import YEARS, CHAPTERS, FLOW, PARTNER, BLOCK_DIR, YEAR_DIR
from lib.prepare_yearly import prepare_yearly

# False: 取得済みブロックを結合するだけ（APIキー不要）
# True: APIで不足ブロックを取得してから結合する
DOWNLOAD_MISSING = False

# 別のフォルダのブロックを使う場合は、この行を Path("保存場所") に変更。
SOURCE_DIR = BLOCK_DIR
OUTDIR = SOURCE_DIR

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 220)


In [ ]:
if DOWNLOAD_MISSING:
    import os
    from dotenv import load_dotenv
    import comtradeapicall as c

    load_dotenv(Path.home() / ".comtrade_env")
    KEY = (os.environ.get("COMTRADE_KEY") or "").strip()
    if not KEY:
        raise RuntimeError("COMTRADE_KEY が未設定です。READMEのAPIキー登録を確認してください。")


In [ ]:
# 保存する18列（API 側の実際の列名。qty / cifvalue / fobvalue は小文字始まり）
COLS = [
    "refYear", "period", "reporterCode", "reporterDesc", "flowDesc",
    "partnerCode", "partnerDesc", "classificationCode", "cmdCode", "cmdDesc",
    "qtyUnitAbbr", "qty", "altQty", "netWgt", "grossWgt",
    "cifvalue", "fobvalue", "primaryValue",
]

# 分割の判断に使う2つの上限（どちらも実測値）
CAP            = 100_000   # 1リクエストの最大レコード数。超過分は黙って捨てられる
CODE_STR_MAX   = 1_200     # cmdCode 文字列の上限。URL全体が2000文字を超えると弾かれる
PAUSE          = 0.3       # リクエスト間の待機（秒）

if DOWNLOAD_MISSING:
    OUTDIR.mkdir(parents=True, exist_ok=True)
print(f"対象年 : {YEARS[0]}〜{YEARS[-1]}（{len(YEARS)}年）")
print(f"対象類 : 第{CHAPTERS[0]}〜{CHAPTERS[-1]}類（{len(CHAPTERS)}類）")
print(f"取得単位: {len(YEARS) * len(CHAPTERS)} ブロック（年 × 類）")
print(f"相手国 : {'World のみ' if PARTNER == '0' else '全相手国'}")
print("出力先 :", OUTDIR.resolve())

## 2. 必要に応じてAPI取得

`DOWNLOAD_MISSING = False` のときはこの節のAPI処理を飛ばします。


In [ ]:
if DOWNLOAD_MISSING:
    HS_CSV = Path("data") / "hs_codes.csv"
    if HS_CSV.exists():
        HS = pd.read_csv(HS_CSV, dtype={"id": str, "parent": str})
    else:
        HS = c.getReference("cmd:HS")
        Path("data").mkdir(exist_ok=True)
        HS.to_csv(HS_CSV, index=False, encoding="utf-8-sig")

    six = HS[HS["aggrLevel"] == 6].copy()
    six["ch"] = six["id"].astype(str).str[:2]

    CODES = {ch: sorted(six.loc[six["ch"] == ch, "id"].astype(str)) for ch in CHAPTERS}
    total_codes = sum(len(v) for v in CODES.values())

    print(f"対象6桁コード: {total_codes:,} 件")
    print({ch: len(v) for ch, v in CODES.items()})

In [ ]:
if DOWNLOAD_MISSING:
    import contextlib
    import io
    import re


    class QuotaExceeded(RuntimeError):
        """API の呼び出し回数クォータを使い切った。"""


    def _args(codes_str, year, **over):
        a = dict(
            typeCode="C", freqCode="A", clCode="HS", period=year,
            reporterCode=None, cmdCode=codes_str, flowCode=FLOW,
            partnerCode=PARTNER, partner2Code="0", customsCode="C00", motCode="0",
        )
        a.update(over)
        return a


    def _call(fn, *args, **kwargs):
        """API を呼び、ライブラリが標準出力に印字するエラーを拾う。

        comtradeapicall は失敗時に例外ではなく「メッセージを print して None を返す」ため、
        標準出力を捕まえないと原因が分からない。403（クォータ切れ）だけは専用の例外にする。
        """
        buf = io.StringIO()
        with contextlib.redirect_stdout(buf):
            res = fn(*args, **kwargs)
        msg = buf.getvalue().strip()
        if msg:
            if "403" in msg or "call volume quota" in msg:
                m = re.search(r"replenished in ([0-9:]+)", msg)
                raise QuotaExceeded(f"呼び出し回数のクォータ切れ。復活まで {m.group(1) if m else '不明'}")
            print(f"    [API] {msg[:160]}")
        return res


    def count_of(codes_str, year, **over):
        """該当件数を返す。取れなければ None。（検証用。通常の取得では使わない）"""
        r = _call(c.getCountFinalData, KEY, **_args(codes_str, year, **over))
        if r is None or getattr(r, "empty", True):
            return None
        return int(r["count"].iloc[0])


    def fetch_raw(codes_str, year, **over):
        """1リクエスト分を取得する。空なら None。"""
        df = _call(c.getFinalData, KEY, maxRecords=CAP, includeDesc=True,
                   **_args(codes_str, year, **over))
        if df is None or getattr(df, "empty", True):
            return None
        return df


    def fetch_codes(codes, year, label="", depth=0):
        """コード列を必要なだけ分割して取得し、DataFrame のリストを返す。

        件数照会は行わない。打ち切りはちょうど CAP 件で起きるため、
        取得結果が CAP 未満なら欠落なしと判断でき、リクエスト数が半分で済む。
        """
        codes_str = ",".join(codes)
        indent = "  " * (depth + 1)

        # ① URL長で分割
        if len(codes_str) > CODE_STR_MAX and len(codes) > 1:
            mid = len(codes) // 2
            return (fetch_codes(codes[:mid], year, label, depth + 1)
                    + fetch_codes(codes[mid:], year, label, depth + 1))

        df = fetch_raw(codes_str, year)
        time.sleep(PAUSE)
        got = 0 if df is None else len(df)

        # ② 上限ちょうど = 打ち切りの可能性 → 分割して取り直す
        if got >= CAP:
            if len(codes) > 1:
                mid = len(codes) // 2
                print(f"{indent}上限 {CAP:,} 件に到達 → {len(codes)}コードを二分割")
                return (fetch_codes(codes[:mid], year, label, depth + 1)
                        + fetch_codes(codes[mid:], year, label, depth + 1))
            # 単一コードで超過 → 報告国で分ける
            print(f"{indent}単一コード {codes[0]} が上限到達 → 報告国で分割")
            reps = sorted(c.getReference("reporter")["reporterCode"].astype(str).unique())
            out = []
            for i in range(0, len(reps), 40):
                grp = ",".join(reps[i:i + 40])
                d = fetch_raw(codes_str, year, reporterCode=grp)
                time.sleep(PAUSE)
                if d is not None:
                    if len(d) >= CAP:
                        raise RuntimeError(f"{label}: {codes[0]} は報告国分割でも上限に達した")
                    out.append(d)
            return out

        if got:
            print(f"{indent}{len(codes):>3}コード → {got:>7,} 件")
        return [df] if got else []

In [ ]:
if DOWNLOAD_MISSING:
    t_start = time.time()
    summary = []
    blocks = [(y, ch) for y in YEARS for ch in CHAPTERS]
    todo = [(y, ch) for y, ch in blocks if not (OUTDIR / f"ch{ch}_{y}.csv.gz").exists()]

    print(f"全 {len(blocks)} ブロック中 {len(blocks)-len(todo)} 件は取得済み。残り {len(todo)} 件を処理する。\n")
    stopped = None

    for k, (year, ch) in enumerate(todo, 1):
        out_path = OUTDIR / f"ch{ch}_{year}.csv.gz"
        codes = CODES[ch]
        print(f"[{k}/{len(todo)}] {year} 第{ch}類 — {len(codes)}コード")
        t0 = time.time()

        try:
            frames = fetch_codes(codes, year, label=f"{year}/ch{ch}")
        except QuotaExceeded as e:
            stopped = e
            break

        if not frames:
            print("  データなし")
            summary.append((year, ch, 0, 0.0))
            continue

        df_ch = pd.concat(frames, ignore_index=True)

        missing = [col for col in COLS if col not in df_ch.columns]
        if missing:
            raise RuntimeError(f"列が存在しない: {missing}")
        df_ch = df_ch[COLS]

        # hs_codes.csv の id 順（cmdCode 昇順）に並べ替える。
        # API の返却順は報告国コード順で、しかも保証されていないため自前で固定する。
        # 6桁ゼロ埋めの文字列なので、辞書順＝数値順になる。
        df_ch = (df_ch
                 .astype({"cmdCode": str})
                 .sort_values(["cmdCode", "reporterCode", "partnerCode"], kind="stable")
                 .reset_index(drop=True))

        df_ch.to_csv(out_path, index=False, encoding="utf-8-sig", compression="gzip")
        print(f"  → {len(df_ch):,} 行 / {time.time()-t0:.0f}秒 / {out_path.name}")
        summary.append((year, ch, len(df_ch), time.time() - t0))

    done = len(list(OUTDIR.glob("ch*_*.csv.gz")))
    print(f"\n{'='*60}")
    if stopped:
        print(f"■ 中断: {stopped}")
        print("  → クォータ復活後にこのセルを再実行すれば続きから走る。")
    else:
        print("■ 完了")
    print(f"進捗: {done}/{len(blocks)} ブロック（残り {len(blocks)-done}）")
    print(f"経過: {time.time()-t_start:.0f} 秒")
    if summary:
        s = pd.DataFrame(summary, columns=["year", "chapter", "rows", "sec"])
        print(f"今回取得: {s['rows'].sum():,} 行")
        display(s.groupby("year", as_index=False)["rows"].sum())

## 3. 年別CSVを保存

全ブロックが揃っていることを確認してから、1類ずつ年別ファイルに書き込みます。
再実行すると年別ファイルを再作成し、完成後に置き換えます。
API利用枠で中断した場合は、不足ブロックを取得してからこの処理を実行してください。


In [ ]:
prepare_yearly(source=SOURCE_DIR, destination=YEAR_DIR, years=YEARS, chapters=CHAPTERS)


## 4. 出力を確認


In [ ]:
for year in YEARS:
    output_path = YEAR_DIR / f"trade_{year}.csv.gz"
    print(f"{year}: {output_path.name} ({output_path.stat().st_size / 1e6:.1f} MB)")


## メモ

- **`cifvalue` は輸出データではほぼ空**（CIF は輸入側の評価額）。輸出額は `fobvalue`
  または `primaryValue` を使う。
- `partnerDesc == "World"` は全世界合計。個別相手国と足すと**二重計上**になる。
- `partner2Code="0"` / `customsCode="C00"` / `motCode="0"` は合計行だけを取るための指定。
  外すと輸送モード別・通関手続別・原産国別の内訳が混ざって数倍に膨れる。
- 対象年を変えるには `settings.py` の `START_YEAR` / `END_YEAR` を書き換える。
- 途中で止まったら同じセルを再実行すれば、保存済みの類を飛ばして続きから走る。
  特定のブロックをやり直したいときは `OUTDIR` 内の `ch{類}_{年}.csv.gz` を消してから再実行する。